Collection of data ---> Data preprocessing ---> Train Test split ---> Logistic Regression model ---> Trained Logistic Regression model 

1: Fake news
2: Real news 

Importing the Dependencies

In [ ]:
import numpy as np
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [ ]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\vishu\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
#printing the stopwords in english
print(stopwords.words('english'))


['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

In [ ]:
# loading the dataset to a pandas Dataframe
news_dataset = pd.read_csv('FakeNewsNet.csv')

In [ ]:
news_dataset

,title,news_url,source_domain,tweet_num,real
0,Kandi Burruss Explodes Over Rape Accusation on...,http://toofab.com/2017/05/08/real-housewives-a...,toofab.com,42,1
1,People's Choice Awards 2018: The best red carp...,https://www.today.com/style/see-people-s-choic...,www.today.com,0,1
2,Sophia Bush Sends Sweet Birthday Message to 'O...,https://www.etonline.com/news/220806_sophia_bu...,www.etonline.com,63,1
3,Colombian singer Maluma sparks rumours of inap...,https://www.dailymail.co.uk/news/article-33655...,www.dailymail.co.uk,20,1
4,Gossip Girl 10 Years Later: How Upper East Sid...,https://www.zerchoo.com/entertainment/gossip-g...,www.zerchoo.com,38,1
...,...,...,...,...,...
23191,Pippa Middleton wedding: In case you missed it...,https://www.express.co.uk/news/royal/807049/pi...,www.express.co.uk,52,1
23192,Zayn Malik & Gigi Hadid’s Shocking Split: Why ...,hollywoodlife.com/2018/03/13/zayn-malik-gigi-h...,hollywoodlife.com,7,0
23193,Jessica Chastain Recalls the Moment Her Mother...,http://www.justjared.com/2018/01/17/jessica-ch...,www.justjared.com,26,1
23194,"Tristan Thompson Feels ""Dumped"" After Khloé Ka...",www.intouchweekly.com/posts/tristan-thompson-f...,www.intouchweekly.com,24,0


In [ ]:
news_dataset.shape

(23196, 5)

In [ ]:
news_dataset.head()

,title,news_url,source_domain,tweet_num,real
0,Kandi Burruss Explodes Over Rape Accusation on...,http://toofab.com/2017/05/08/real-housewives-a...,toofab.com,42,1
1,People's Choice Awards 2018: The best red carp...,https://www.today.com/style/see-people-s-choic...,www.today.com,0,1
2,Sophia Bush Sends Sweet Birthday Message to 'O...,https://www.etonline.com/news/220806_sophia_bu...,www.etonline.com,63,1
3,Colombian singer Maluma sparks rumours of inap...,https://www.dailymail.co.uk/news/article-33655...,www.dailymail.co.uk,20,1
4,Gossip Girl 10 Years Later: How Upper East Sid...,https://www.zerchoo.com/entertainment/gossip-g...,www.zerchoo.com,38,1


In [ ]:
news_dataset.isnull().sum()

title              0
news_url         330
source_domain    330
tweet_num          0
real               0
dtype: int64

In [ ]:
news_dataset = news_dataset.fillna('')

In [ ]:
news_dataset['content'] = news_dataset['source_domain']+''+news_dataset['title']


In [ ]:
print(news_dataset['content'])

0        toofab.comKandi Burruss Explodes Over Rape Acc...
1        www.today.comPeople's Choice Awards 2018: The ...
2        www.etonline.comSophia Bush Sends Sweet Birthd...
3        www.dailymail.co.ukColombian singer Maluma spa...
4        www.zerchoo.comGossip Girl 10 Years Later: How...
                               ...                        
23191    www.express.co.ukPippa Middleton wedding: In c...
23192    hollywoodlife.comZayn Malik & Gigi Hadid’s Sho...
23193    www.justjared.comJessica Chastain Recalls the ...
23194    www.intouchweekly.comTristan Thompson Feels "D...
23195    www.billboard.comKelly Clarkson Performs a Med...
Name: content, Length: 23196, dtype: object


In [ ]:
# separating the data & label
X = news_dataset.drop(columns='news_url',axis = 1)
Y = news_dataset['news_url']

In [ ]:
print(X)
print(Y)

                                                   title  \
0      Kandi Burruss Explodes Over Rape Accusation on...   
1      People's Choice Awards 2018: The best red carp...   
2      Sophia Bush Sends Sweet Birthday Message to 'O...   
3      Colombian singer Maluma sparks rumours of inap...   
4      Gossip Girl 10 Years Later: How Upper East Sid...   
...                                                  ...   
23191  Pippa Middleton wedding: In case you missed it...   
23192  Zayn Malik & Gigi Hadid’s Shocking Split: Why ...   
23193  Jessica Chastain Recalls the Moment Her Mother...   
23194  Tristan Thompson Feels "Dumped" After Khloé Ka...   
23195  Kelly Clarkson Performs a Medley of Kendrick L...   

               source_domain  tweet_num  real  \
0                 toofab.com         42     1   
1              www.today.com          0     1   
2           www.etonline.com         63     1   
3        www.dailymail.co.uk         20     1   
4            www.zerchoo.com      

Stemming:
    Stemming is the process of reducing a word to its Root word
    Example: actor, actress, acting --> act

In [ ]:
port_stem = PorterStemmer()

In [ ]:
def stemming(content):
    stemmed_content = re.sub('[^a-zA-Z]','',content)
    stemmed_content = stemmed_content.lower()
    stemmed_content = stemmed_content.split()
    stemmed_content = [port_stem.stem(word) for word in stemmed_content if not word in stopwords.words('english')]
    stemmed_content = ''.join(stemmed_content)
    return stemmed_content

In [ ]:
news_dataset['content']=news_dataset['content'].apply(stemming)


In [ ]:
print(news_dataset['content'])

0        toofabcomkandiburrussexplodesoverrapeaccusatio...
1        wwwtodaycompeopleschoiceawardsthebestredcarpet...
2        wwwetonlinecomsophiabushsendssweetbirthdaymess...
3        wwwdailymailcoukcolombiansingermalumasparksrum...
4        wwwzerchoocomgossipgirlyearslaterhowuppereasts...
                               ...                        
23191    wwwexpresscoukpippamiddletonweddingincaseyoumi...
23192    hollywoodlifecomzaynmalikgigihadidsshockingspl...
23193    wwwjustjaredcomjessicachastainrecallsthemoment...
23194    wwwintouchweeklycomtristanthompsonfeelsdumpeda...
23195    wwwbillboardcomkellyclarksonperformsamedleyofk...
Name: content, Length: 23196, dtype: object


In [ ]:
# separating the data and label
X = news_dataset['content'].values
Y = news_dataset['news_url'].values


In [ ]:
print(X)

['toofabcomkandiburrussexplodesoverrapeaccusationonrealhousewivesofatlantareunionvideo'
 'wwwtodaycompeopleschoiceawardsthebestredcarpetlook'
 'wwwetonlinecomsophiabushsendssweetbirthdaymessagetoonetreehillcostarhilarieburtonbreytoneva'
 ...
 'wwwjustjaredcomjessicachastainrecallsthemomenthermothersboyfriendslappedherijustkickedhiminthegenit'
 'wwwintouchweeklycomtristanthompsonfeelsdumpedafterkhlokardashianrefusestolethimmoveintolahomeexclus'
 'wwwbillboardcomkellyclarksonperformsamedleyofkendricklamarshumblemorehitsatthebillboardmusicaward']


In [ ]:
print(Y)

['http://toofab.com/2017/05/08/real-housewives-atlanta-kandi-burruss-rape-phaedra-parks-porsha-williams/'
 'https://www.today.com/style/see-people-s-choice-awards-red-carpet-looks-t141832'
 'https://www.etonline.com/news/220806_sophia_bush_sends_sweet_birthday_message_to_one_tree_hill_co_star_hilarie_burton_breyton_4eva'
 ...
 'http://www.justjared.com/2018/01/17/jessica-chastain-recalls-the-moment-her-mothers-boyfriend-slapped-her-i-just-kicked-him-in-the-genitals/'
 'www.intouchweekly.com/posts/tristan-thompson-feels-dumped-khloe-kardashian-162092'
 'https://www.billboard.com/articles/news/bbma/8456910/kelly-clarkson-medley-bbmas']


In [ ]:
Y.shape

(23196,)

In [ ]:
vectorizer = TfidfVectorizer()
vectorizer.fit(X)

X = vectorizer.transform(X)

AttributeError: 'csr_matrix' object has no attribute 'lower'

In [ ]:
print(X)

  (0, 7227)	1.0
  (1, 19409)	1.0
  (2, 12549)	1.0
  (3, 10629)	1.0
  (4, 21621)	1.0
  (5, 14720)	1.0
  (6, 21687)	1.0
  (7, 11996)	1.0
  (8, 8501)	1.0
  (9, 17240)	1.0
  (10, 20673)	1.0
  (11, 21703)	1.0
  (12, 6083)	1.0
  (13, 24)	1.0
  (14, 7808)	1.0
  (15, 11906)	1.0
  (16, 18863)	1.0
  (17, 19740)	1.0
  (18, 8016)	1.0
  (19, 2866)	1.0
  (20, 709)	1.0
  (21, 17988)	1.0
  (22, 10823)	1.0
  (23, 2728)	1.0
  (24, 16285)	1.0
  :	:
  (23171, 9969)	1.0
  (23172, 13884)	1.0
  (23173, 19395)	1.0
  (23174, 12729)	1.0
  (23175, 21095)	1.0
  (23176, 19011)	1.0
  (23177, 13829)	1.0
  (23178, 8308)	1.0
  (23179, 20110)	1.0
  (23180, 2935)	1.0
  (23181, 5390)	1.0
  (23182, 7079)	1.0
  (23183, 14447)	1.0
  (23184, 2311)	1.0
  (23185, 4826)	1.0
  (23186, 19723)	1.0
  (23187, 17157)	1.0
  (23188, 12867)	1.0
  (23189, 4569)	1.0
  (23190, 17937)	1.0
  (23191, 12658)	1.0
  (23192, 3172)	1.0
  (23193, 14902)	1.0
  (23194, 14773)	1.0
  (23195, 8808)	1.0


In [ ]:
print(Y)

['http://toofab.com/2017/05/08/real-housewives-atlanta-kandi-burruss-rape-phaedra-parks-porsha-williams/'
 'https://www.today.com/style/see-people-s-choice-awards-red-carpet-looks-t141832'
 'https://www.etonline.com/news/220806_sophia_bush_sends_sweet_birthday_message_to_one_tree_hill_co_star_hilarie_burton_breyton_4eva'
 ...
 'http://www.justjared.com/2018/01/17/jessica-chastain-recalls-the-moment-her-mothers-boyfriend-slapped-her-i-just-kicked-him-in-the-genitals/'
 'www.intouchweekly.com/posts/tristan-thompson-feels-dumped-khloe-kardashian-162092'
 'https://www.billboard.com/articles/news/bbma/8456910/kelly-clarkson-medley-bbmas']


Splitting the dataset to training & test data

In [ ]:
X_train,X_test,Y_train,Y_test = train_test_split(X,Y,test_size = 0.2,stratify = Y, random_state = 2)


ValueError: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.

In [ ]:
model = LogisticRegression()


In [ ]:
model.fit(X_train,Y_train)

Evaluation:

Accuracy score

In [ ]:
# accuracy score on the training data
X_train_prediction = model.predict(X_train)
training_data_accuracy = accuracy_score(X_train_prediction, Y_train)

In [ ]:
print("Accuracy score of the training data:",training_data_accuracy)